In [1]:
import cartopy.crs as ccrs
import xarray as xr
import pandas as pd
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from matplotlib import pyplot as plt
import glob
import matplotlib
%matplotlib inline

In [2]:
#define focusing area
[min_lon, max_lon, min_lat, max_lat]=[100.,150.,10.,50.]
[min_lon_tw, max_lon_tw, min_lat_tw, max_lat_tw]=[118,126,22,26]
[min_lon_ish, max_lon_ish, min_lat_ish, max_lat_ish]=[122,126,22,26]
slp_levels=np.linspace(980,1040,16)
slp_lws=[2 if (pp-980)%20==0 else 1 for pp in slp_levels]
prec_threshold=0.1
ishigaki={'lat':24.33,'lon':124.16}

In [3]:
dataDir='/data/mileshsieh/CMIP6'
m='TaiESM1'
sce='historical'

template='%s/%s/%s/atmos/day/r1i1p1f1/%s_day_%s_%s_r1i1p1f1_gn_%04d0101-*.nc'
varList=['psl','ua','va','pr']
p_selected=1000

In [4]:
#LTS
ds_model_ta=xr.open_mfdataset([glob.glob(template%(dataDir,m,sce,'ta',m,sce,yr_start))[0] for yr_start in [2000,2010]])\
                              .sel(lat=slice(min_lat,max_lat),lon=slice(min_lon,max_lon),
                                            time=slice('2005-01-01', '2014-12-31'),plev=[70000,100000])
ds_model_ta['LTS']=ds_model_ta.ta.sel(plev=70000)*np.power(1000/700,287/1004)-ds_model_ta.ta.sel(plev=100000).compute()
ds_model_ta

<xarray.Dataset>
Dimensions:    (time: 3650, bnds: 2, plev: 2, lat: 42, lon: 41)
Coordinates:
  * time       (time) object 2005-01-01 12:00:00 ... 2014-12-31 12:00:00
  * plev       (plev) float64 7e+04 1e+05
  * lat        (lat) float64 10.84 11.78 12.72 13.66 ... 46.65 47.59 48.53 49.48
  * lon        (lon) float64 100.0 101.2 102.5 103.8 ... 146.2 147.5 148.8 150.0
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object dask.array<chunksize=(1825, 2), meta=np.ndarray>
    lat_bnds   (time, lat, bnds) float64 dask.array<chunksize=(1825, 42, 2), meta=np.ndarray>
    lon_bnds   (time, lon, bnds) float64 dask.array<chunksize=(1825, 41, 2), meta=np.ndarray>
    ta         (time, plev, lat, lon) float32 dask.array<chunksize=(1825, 2, 42, 41), meta=np.ndarray>
    LTS        (time, lat, lon) float32 dask.array<chunksize=(1825, 42, 41), meta=np.ndarray>
Attributes: (12/50)
    Conventions:               CF-1.7 CMIP-6.2
    activity_id:               CMIP
    branch_method:             Hybrid-restart from year 0671-01-01 of piControl
    branch_time:               0.0
    branch_time_in_child:      0.0
    branch_time_in_parent:     171550.0
    ...                        ...
    title:                     TaiESM1 output prepared for CMIP6
    variable_id:               ta
    variant_label:             r1i1p1f1
    license:                   CMIP6 model data produced by NCC is licensed u...
    cmor_version:              3.5.0
    tracking_id:               hdl:21.14100/bc0bf560-b548-49b9-bc56-fe74331f7d20

In [5]:
#open mfdataset (multiple files dataset)
fList=[]
for yr_start in [2000,2010]:
    #open multiple files
    fList=fList+[glob.glob(template%(dataDir,m,sce,var,m,sce,yr_start))[0] for var in varList]

fList

['/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/psl_day_TaiESM1_historical_r1i1p1f1_gn_20000101-20091231.nc',
 '/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/ua_day_TaiESM1_historical_r1i1p1f1_gn_20000101-20091231.nc',
 '/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/va_day_TaiESM1_historical_r1i1p1f1_gn_20000101-20091231.nc',
 '/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/pr_day_TaiESM1_historical_r1i1p1f1_gn_20000101-20091231.nc',
 '/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/psl_day_TaiESM1_historical_r1i1p1f1_gn_20100101-20141231.nc',
 '/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/ua_day_TaiESM1_historical_r1i1p1f1_gn_20100101-20141231.nc',
 '/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/va_day_TaiESM1_historical_r1i1p1f1_gn_20100101-20141231.nc',
 '/data/mileshsieh/CMIP6/TaiESM1/historical/atmos/day/r1i1p1f1/pr_day_TaiESM1_historical_r1i1p1f1_gn_20100101-20141231.nc']

In [6]:
#open multiple files
ds_model_uv_mslp=xr.open_mfdataset(fList).sel(lat=slice(min_lat,max_lat),lon=slice(min_lon,max_lon),
                                            time=slice('2005-01-01', '2014-12-31'),plev=p_selected*100)

ds_model=xr.merge([ds_model_uv_mslp,ds_model_ta.LTS])
#convert time coordinate to date
ds_model.coords['time'] = ds_model.time.dt.floor('1D')
ds_model

<xarray.Dataset>
Dimensions:    (time: 3650, bnds: 2, lat: 42, lon: 41)
Coordinates:
  * time       (time) object 2005-01-01 00:00:00 ... 2014-12-31 00:00:00
  * lat        (lat) float64 10.84 11.78 12.72 13.66 ... 46.65 47.59 48.53 49.48
  * lon        (lon) float64 100.0 101.2 102.5 103.8 ... 146.2 147.5 148.8 150.0
    plev       float64 1e+05
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object dask.array<chunksize=(1825, 2), meta=np.ndarray>
    lat_bnds   (time, lat, bnds) float64 dask.array<chunksize=(1825, 42, 2), meta=np.ndarray>
    lon_bnds   (time, lon, bnds) float64 dask.array<chunksize=(1825, 41, 2), meta=np.ndarray>
    pr         (time, lat, lon) float32 dask.array<chunksize=(1825, 42, 41), meta=np.ndarray>
    psl        (time, lat, lon) float32 dask.array<chunksize=(1825, 42, 41), meta=np.ndarray>
    ua         (time, lat, lon) float32 dask.array<chunksize=(1825, 42, 41), meta=np.ndarray>
    va         (time, lat, lon) float32 dask.array<chunksize=(1825, 42, 41), meta=np.ndarray>
    LTS        (time, lat, lon) float32 dask.array<chunksize=(1825, 42, 41), meta=np.ndarray>
Attributes: (12/50)
    Conventions:               CF-1.7 CMIP-6.2
    activity_id:               CMIP
    branch_method:             Hybrid-restart from year 0671-01-01 of piControl
    branch_time:               0.0
    branch_time_in_child:      0.0
    branch_time_in_parent:     171550.0
    ...                        ...
    title:                     TaiESM1 output prepared for CMIP6
    variable_id:               pr
    variant_label:             r1i1p1f1
    license:                   CMIP6 model data produced by NCC is licensed u...
    cmor_version:              3.5.0
    tracking_id:               hdl:21.14100/a6c794cd-eba3-4a84-b546-905f4eb93ed2

In [7]:
def uv2ws(u, v):
    func = lambda x, y: np.sqrt(x**2 + y**2)
    return xr.apply_ufunc(func, u, v, dask="allowed")

def uv2wd(u, v):
    func = lambda x,y: np.where(u==0,np.where(v>0,180.0,0.0),180.0 + np.arctan2(u, v) * 180.0 / np.pi)
    return xr.apply_ufunc(func, u, v, dask="allowed")

def calc_wd_cor(wd_array):
    wd_min=wd_array.min()
    #print(wd_min)
    wdList=np.array([dd-wd_min if abs(dd-wd_min)<180.0 else wd_min-(dd-360.0) for dd in wd_array.ravel()])
    result=np.percentile(wdList,75)
    return result
def uv2deg_ish(ser):
    if ser.ish_u==0:
        return 180.0 if ser.ish_v>0 else 0.0
    else:
        tmpdir=270-np.arctan(ser.ish_v/ser.ish_u)/np.pi*180
        return tmpdir if ser.ish_u>0 else tmpdir-180
    
def uv2deg_mean(ser):
    if ser.meanU==0:
        return 180.0 if ser.meanV>0 else 0.0
    else:
        tmpdir=270-np.arctan(ser.meanV/ser.meanU)/np.pi*180
        return tmpdir if ser.meanU>0 else tmpdir-180

In [8]:

tw_ds=ds_model.sel(lat=slice(min_lat_tw,max_lat_tw),lon=slice(min_lon_tw,max_lon_tw))
tw_ds['precipitation']=tw_ds.pr*3600
tw_ds['prRatio']=tw_ds.precipitation.where(tw_ds.precipitation>=prec_threshold).count(dim=['lon','lat'])/30.0
df_tw=pd.DataFrame({'date':tw_ds.time.values,'prRatio':tw_ds.prRatio.values})
    
ish_ds=ds_model.sel(lat=slice(min_lat_ish,max_lat_ish),lon=slice(min_lon_ish,max_lon_ish))
ish_ds['ws']=uv2ws(ish_ds.ua, ish_ds.va)
ish_ds['wd']=uv2wd(ish_ds.ua, ish_ds.va)
#ish_ds['wd_p90']=ish_ds.wd.reduce(np.percentile, q=90,dim=['lat','lon'])
#ish_ds['wd_p10']=ish_ds.wd.reduce(np.percentile, q=10,dim=['lat','lon'])
#ish_ds['wdCor_raw']=ish_ds.wd_p90-ish_ds.wd_p10
#ish_ds['wdCor']=ish_ds.wdCor_raw.where(ish_ds.wdCor_raw<180,360-ish_ds.wdCor_raw)
ish_ds['wdCor']=xr.apply_ufunc(calc_wd_cor,ish_ds.wd,input_core_dims=[['lat','lon']],
                                   output_core_dims=[[]],vectorize=True)
ish_ds['ws_mean']=ish_ds.ws.reduce(np.mean,dim=['lat','lon'])
ish_ds['maxWS']=ish_ds.ws.reduce(np.max,dim=['lat','lon'])
ish_ds['u_mean']=ish_ds.ua.reduce(np.mean,dim=['lat','lon'])
ish_ds['v_mean']=ish_ds.va.reduce(np.mean,dim=['lat','lon'])  
ish_ds['LTS_mean']=ish_ds.LTS.reduce(np.mean,dim=['lat','lon'])  
idx=ish_ds.wdCor.idxmax(dim='time')
dateStr=ish_ds.wdCor.idxmax(dim='time').dt.strftime('%Y-%m-%d').item()
print(dateStr,pd.to_datetime(dateStr).dayofyear,ish_ds.wdCor.max().item())
#save to dataframe
yyyymmdd=ish_ds.time.dt.strftime('%Y%m%d')
u_ish=ish_ds.ua.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
v_ish=ish_ds.va.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
LTS_ish=ish_ds.LTS.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
wdCor=ish_ds.wdCor.values
df_ish=pd.DataFrame({'date':ish_ds.time.values,'yyyymmdd':yyyymmdd, 
                     'meanU':ish_ds.u_mean.values,'meanV':ish_ds.v_mean.values,'meanLTS':ish_ds.LTS_mean.values,
                     'wdCor':wdCor,'meanWS':ish_ds.ws_mean.values,'maxWS':ish_ds.maxWS.values,
                     'ish_u':u_ish,'ish_v':v_ish,'ish_LTS':LTS_ish})
df_ish['meanWD']=df_ish.apply(uv2deg_mean,axis=1)
df_ish['ish_wd']=df_ish.apply(uv2deg_ish,axis=1)
df_ish['ish_ws']=df_ish.apply(lambda ser: np.sqrt(ser.ish_u*ser.ish_u+ser.ish_v*ser.ish_v),axis=1)
df_ish['year']=df_ish.apply(lambda ser: ser.date.year,axis=1)
df_ish['month']=df_ish.apply(lambda ser: ser.date.month,axis=1)
    
final=pd.merge(df_ish,df_tw,on='date')


2011-08-29 241 177.10635375976562


In [9]:
final[final.month.isin([1,2,3,4,10,11,12])].to_csv('../data/cold_season_%s_%s.indices.csv'%(m,sce), index=False)
final

,date,yyyymmdd,meanU,meanV,meanLTS,wdCor,meanWS,maxWS,ish_u,ish_v,ish_LTS,meanWD,ish_wd,ish_ws,year,month,prRatio
0,2005-01-01 00:00:00,20050101,-10.702707,3.062799,16.068916,47.683964,12.057169,13.968800,-10.882577,3.729663,16.538622,105.969558,108.917550,11.503950,2005,1,0.000000
1,2005-01-02 00:00:00,20050102,-3.446028,1.973722,16.980869,92.500526,5.464157,8.322924,-3.019558,2.745460,17.643442,119.802044,132.277921,4.081089,2005,1,0.000000
2,2005-01-03 00:00:00,20050103,-4.222742,-8.530740,16.052675,23.146568,9.711199,13.056018,-3.408729,-8.634366,15.636638,26.335547,21.543424,9.282872,2005,1,0.433333
3,2005-01-04 00:00:00,20050104,-6.793808,-12.298906,14.006342,19.369743,14.217945,15.466806,-6.902154,-13.006863,13.097582,28.915877,27.952869,14.724748,2005,1,0.533333
4,2005-01-05 00:00:00,20050105,-4.598643,-10.663625,14.625655,25.384926,11.864710,15.080824,-3.852344,-11.200743,15.310399,23.327916,18.979960,11.844712,2005,1,0.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3645,2014-12-27 00:00:00,20141227,-4.539152,-4.490469,14.420172,19.486328,6.412172,10.802748,-4.009673,-4.475425,14.339736,45.308901,41.858160,6.008902,2014,12,0.000000
3646,2014-12-28 00:00:00,20141228,-5.618223,-4.369158,12.129134,21.101410,7.152671,11.428333,-5.598631,-4.465687,11.789370,52.128647,51.422769,7.161497,2014,12,0.000000
3647,2014-12-29 00:00:00,20141229,-6.605102,-7.273139,10.164942,25.644966,9.942562,13.192649,-5.792758,-7.621244,9.386062,42.244164,37.237755,9.572847,2014,12,0.000000
3648,2014-12-30 00:00:00,20141230,-4.795290,-4.263477,9.780099,17.513363,6.460807,9.811886,-4.290940,-4.280844,8.895932,48.359805,45.067484,6.061170,2014,12,0.000000
